# Experiment 9: Logit Adjustment Sweep — Post-hoc Shirt Bias

**Single variable changed**: inference-time logit bias added to Shirt class
**Held constant**: architecture (DiagnosticCNN), loss (CrossEntropy), optimizer (Adam lr=0.001), epochs (40 with cosine LR), data (no augmentation)

## Rationale

E8 (extended training) achieved the highest overall accuracy (92.99%) and best Shirt Precision (0.792) but Shirt TPR dropped to 0.797. Adding a positive bias to the Shirt logit at inference should shift the decision boundary to favor Shirt, recovering recall at the cost of precision. Sweeping the bias value identifies the optimal operating point on Shirt's precision–recall curve — zero retraining cost after the model is trained.

## Expected behavior

| Bias direction | Effect on Shirt | Use case |
|:--------------:|----------------|----------|
| Negative (< 0) | Fewer Shirt predictions (lower TPR, higher Prec) | Conservative: avoid Shirt unless certain |
| Zero (= 0) | Default E8 behavior | Original converged model |
| Positive (> 0) | More Shirt predictions (higher TPR, lower Prec) | Aggressive: catch more true Shirts |

We expect positive bias to behave analogously to E5/E7 — improving Shirt TPR at the cost of other upper-body classes and overall accuracy.

In [1]:
import sys; sys.path.append('..')

import os, torch, torch.nn as nn, torch.optim as optim
import numpy as np
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from src.data_utils import load_fashionmnist, get_dataloaders
from src.train_utils import train_one_epoch

OUT_DIR = '../outputs/error_analysis/logit_adjustment'
os.makedirs(OUT_DIR, exist_ok=True)

print(f'PyTorch: {torch.__version__}')
if torch.backends.mps.is_available():    device = 'mps'
elif torch.cuda.is_available():          device = 'cuda'
else:                                    device = 'cpu'
print(f'Device: {device}')

PyTorch: 2.13.0+cu130
Device: cuda


## Dataset — identical to E1/E8 (no augmentation)

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
train_ds = __import__('torchvision').datasets.FashionMNIST(root='../data', train=True, download=True, transform=transform)
test_ds  = __import__('torchvision').datasets.FashionMNIST(root='../data', train=False, download=True, transform=transform)
class_names = train_ds.classes

train_loader, test_loader = get_dataloaders(train_ds, test_ds, batch_size=64)
print(f'Train: {len(train_loader)} batches, Test: {len(test_loader)} batches')

Train: 938 batches, Test: 157 batches


## Architecture — DiagnosticCNN (identical to E1)

In [3]:
class DiagnosticCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1); self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1); self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1); self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1); self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.conv5 = nn.Conv2d(64, 128, 3, padding=1); self.bn5 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(128, num_classes)
        self.relu = nn.ReLU(inplace=True)

    def get_features(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.relu(self.bn5(self.conv5(x)))
        x = self.pool3(x)
        x = self.gap(x)
        return x.view(x.size(0), -1)

    def forward(self, x):
        x = self.get_features(x)
        x = self.drop(x)
        x = self.fc(x)
        return x

model = DiagnosticCNN().to(device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

Params: 140,778


c:\document\Study documents\Deeplearning_Course\.venv\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


## Training — 40 epochs with cosine LR (identical to E8)

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = CosineAnnealingLR(optimizer, T_max=40)
EPOCHS = 40

train_losses = []
model.train()
for epoch in range(EPOCHS):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(loss)
    scheduler.step()
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}] Loss: {loss:.4f}')

with open(os.path.join(OUT_DIR, 'train_losses.txt'), 'w') as f:
    for l in train_losses:
        f.write(f'{l}\n')

# Save model weights for logit adjustment experiments
torch.save(model.state_dict(), os.path.join(OUT_DIR, 'model_weights.pth'))
print(f'Weights saved. Final loss: {train_losses[-1]:.4f}')

Epoch [1/40] Loss: 0.4593
Epoch [10/40] Loss: 0.1340
Epoch [20/40] Loss: 0.0423
Epoch [30/40] Loss: 0.0092
Epoch [40/40] Loss: 0.0032
Weights saved. Final loss: 0.0032


## Evaluation with logit bias sweep

In [5]:
from src.eval_utils import get_all_probas_and_labels
from sklearn.metrics import accuracy_score, confusion_matrix

SHIRT_IDX = 6
BIASES = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]

all_results = []
model.eval()

with torch.no_grad():
    for bias in BIASES:
        all_preds, all_labels = [], []
        sh_tp = sh_fp = sh_fn = 0

        for inputs, lbls in test_loader:
            inputs, lbls = inputs.to(device), lbls.to(device)
            logits = model(inputs)
            logits[:, SHIRT_IDX] += bias
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())

            # Per-class counts
            for true, pred in zip(lbls.cpu().numpy(), preds.cpu().numpy()):
                if pred == SHIRT_IDX and true == SHIRT_IDX: sh_tp += 1
                if pred == SHIRT_IDX and true != SHIRT_IDX: sh_fp += 1
                if pred != SHIRT_IDX and true == SHIRT_IDX: sh_fn += 1

        acc = accuracy_score(all_labels, all_preds)
        cm = confusion_matrix(all_labels, all_preds)

        tpr_sh = sh_tp / (sh_tp + sh_fn + 1e-8)
        prec_sh = sh_tp / (sh_tp + sh_fp + 1e-8)

        # Per-class TPR
        per_class_tpr = {}
        for i, name in enumerate(class_names):
            tp = cm[i, i]
            fn = cm[i].sum() - tp
            per_class_tpr[name] = tp / (tp + fn + 1e-8)

        all_results.append({
            'bias': bias,
            'acc': round(acc * 100, 2),
            'shirt_tpr': round(tpr_sh, 4),
            'shirt_prec': round(prec_sh, 4),
            'per_class_tpr': per_class_tpr,
            'sh_fp': sh_fp,
            'sh_fn': sh_fn,
        })
        print(f'bias={bias:+.1f}  acc={acc*100:.2f}%  Shirt TPR={tpr_sh:.4f}  Shirt Prec={prec_sh:.4f}')

bias=-1.0  acc=93.17%  Shirt TPR=0.7490  Shirt Prec=0.8285
bias=-0.5  acc=93.21%  Shirt TPR=0.7660  Shirt Prec=0.8166
bias=+0.0  acc=93.26%  Shirt TPR=0.7850  Shirt Prec=0.8060
bias=+0.5  acc=93.25%  Shirt TPR=0.7990  Shirt Prec=0.7934
bias=+1.0  acc=93.17%  Shirt TPR=0.8110  Shirt Prec=0.7776
bias=+1.5  acc=93.11%  Shirt TPR=0.8260  Shirt Prec=0.7634
bias=+2.0  acc=93.00%  Shirt TPR=0.8340  Shirt Prec=0.7507


## Results table

In [6]:
header = f'{"Bias":>6} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10} {"T-sh":>6} {"Pull":>6} {"Coat":>6} {"Dress":>6} {"Trou":>6} {"Sand":>6} {"Snea":>6} {"Bag":>6} {"Boot":>6}'
sep = '-' * len(header)
print(header)
print(sep)

class_short = ['T-sh','Trou','Pull','Dress','Coat','Sand','Shirt','Snea','Bag','Boot']
lines = [header, sep]
for r in all_results:
    tprs = [f'{r["per_class_tpr"][class_names[i]]:.3f}' for i in [0,1,2,3,4,5,7,8,9]]
    line = f'{r["bias"]:>+5.1f} {r["acc"]:>7.2f} {r["shirt_tpr"]:>9.4f} {r["shirt_prec"]:>10.4f} {" ".join(tprs)}'
    lines.append(line)
    print(line)

with open(os.path.join(OUT_DIR, 'bias_sweep_results.txt'), 'w') as f:
    f.write('\n'.join(lines) + '\n')

print(f'\nResults saved to {OUT_DIR}/')

  Bias    Acc%  ShirtTPR  ShirtPrec   T-sh   Pull   Coat  Dress   Trou   Sand   Snea    Bag   Boot
--------------------------------------------------------------------------------------------------
 -1.0   93.17    0.7490     0.8285 0.900 0.988 0.905 0.945 0.903 0.987 0.977 0.990 0.973
 -0.5   93.21    0.7660     0.8166 0.893 0.988 0.903 0.943 0.901 0.987 0.977 0.990 0.973
 +0.0   93.26    0.7850     0.8060 0.886 0.988 0.899 0.942 0.900 0.987 0.977 0.989 0.973
 +0.5   93.25    0.7990     0.7934 0.876 0.988 0.897 0.940 0.899 0.987 0.977 0.989 0.973
 +1.0   93.17    0.8110     0.7776 0.868 0.988 0.893 0.936 0.895 0.987 0.977 0.989 0.973
 +1.5   93.11    0.8260     0.7634 0.860 0.988 0.889 0.931 0.891 0.987 0.977 0.989 0.973
 +2.0   93.00    0.8340     0.7507 0.854 0.988 0.883 0.927 0.888 0.987 0.977 0.989 0.973

Results saved to ../outputs/error_analysis/logit_adjustment/


## Comparison with best prior experiments

In [7]:
print(f'{"Experiment":<20} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}')
print('-' * 48)
# Find the best trade-off from bias sweep
best_acc_r = max(all_results, key=lambda r: r['acc'])
best_shirt_r = max(all_results, key=lambda r: r['shirt_tpr'])
trade_off = [(r['acc'] + r['shirt_tpr'] * 100, r) for r in all_results]
best_trade_r = max(trade_off, key=lambda x: x[0])[1]
print(f'{"E8 (no bias)":<20} {all_results[2]["acc"]:>7.2f} {all_results[2]["shirt_tpr"]:>9.4f} {all_results[2]["shirt_prec"]:>10.4f}')
print(f'{"E7 (label smooth)":<20} {"92.04":>7} {"0.8790":>9} {"0.6921":>10}')
print(f'{"E5 (weighted CE)":<20} {"91.75":>7} {"0.8590":>9} {"0.7058":>10}')
print(f'{"E1 (baseline)":<20} {"92.50":>7} {"0.8470":>9} {"0.7227":>10}')
print(f'{"E9 best trade-off":<20} {best_trade_r["acc"]:>7.2f} {best_trade_r["shirt_tpr"]:>9.4f} {best_trade_r["shirt_prec"]:>10.4f}')
print(f'{"E9 best Shirt TPR":<20} {best_shirt_r["acc"]:>7.2f} {best_shirt_r["shirt_tpr"]:>9.4f} {best_shirt_r["shirt_prec"]:>10.4f}')

Experiment              Acc%  ShirtTPR  ShirtPrec
------------------------------------------------
E8 (no bias)           93.26    0.7850     0.8060
E7 (label smooth)      92.04    0.8790     0.6921
E5 (weighted CE)       91.75    0.8590     0.7058
E1 (baseline)          92.50    0.8470     0.7227
E9 best trade-off      93.00    0.8340     0.7507
E9 best Shirt TPR      93.00    0.8340     0.7507
